In [1]:
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from useful_functions import *
import pandas as pd
from joblib import Parallel, delayed

In [2]:
dp_sample = pd.read_csv('./dp_samples.csv')

In [3]:
dp_sample.drop(columns=[col for col in dp_sample.columns if col.endswith('_dp')], inplace=True)
dp_sample.drop(columns=[col for col in dp_sample.columns if col.endswith('_rank')], inplace=True)
dp_sample.drop(columns=[col for col in dp_sample.columns if col.endswith('dp_count')], inplace=True)
dp_sample.drop(columns=['p_value'], inplace=True)

In [4]:
dp_sample.head()

,TARGETID,dv_r,dv_l,delta_dv,sigma_r,sigma_l,LOGSFR,LOGM
0,39627739380056564,54.235580,-77.214870,131.450450,50.133639,31.151569,0.941032,10.452706
1,39627739380057506,120.705683,-57.385546,178.091228,39.709895,73.005164,-0.880754,10.324258
2,39627739380058467,87.790162,-28.505013,116.295175,30.665694,51.626586,1.031611,10.402426
3,39627739380060210,76.571222,-90.530039,167.101261,52.715443,54.794006,0.039087,10.496312
4,39627739380061198,88.170157,-73.633545,161.803702,38.336279,50.796152,0.903058,9.514435


In [5]:
dp_sample['Regime'] = np.full(dp_sample.shape[0], '0')

In [6]:
dp_sample['Regime'][dp_sample['TARGETID'].isin(read_ids('dp_samples_regime1_ids.txt'))] = '1'
dp_sample['Regime'][dp_sample['TARGETID'].isin(read_ids('dp_samples_regime2_ids.txt'))] = '2'
dp_sample['Regime'][dp_sample['TARGETID'].isin(read_ids('dp_samples_regime3_ids.txt'))] = '3'
dp_sample['Regime'][dp_sample['TARGETID'].isin(read_ids('dp_samples_regime4_ids.txt'))] = '4'

/var/folders/_b/sl_t4k5539781f29qf723b080000gn/T/ipykernel_2370/2724066492.py:1: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  dp_sample['Regime'][dp_sample['TARGETID'].isin(read_ids('dp_samples_regime1_ids.txt'))] = '1'
/var/folders/_b/sl_t

In [7]:
dp_sample.head(10)

,TARGETID,dv_r,dv_l,delta_dv,sigma_r,sigma_l,LOGSFR,LOGM,Regime
0,39627739380056564,54.235580,-77.214870,131.450450,50.133639,31.151569,0.941032,10.452706,1
1,39627739380057506,120.705683,-57.385546,178.091228,39.709895,73.005164,-0.880754,10.324258,4
2,39627739380058467,87.790162,-28.505013,116.295175,30.665694,51.626586,1.031611,10.402426,1
3,39627739380060210,76.571222,-90.530039,167.101261,52.715443,54.794006,0.039087,10.496312,3
4,39627739380061198,88.170157,-73.633545,161.803702,38.336279,50.796152,0.903058,9.514435,3
5,39627739380062010,56.355071,-28.008179,84.363251,42.716501,63.674168,0.143419,10.082529,1
6,39627739384251743,47.120539,-112.720079,159.840618,93.584412,0.010000,0.461850,9.779942,4
7,39627739384252917,53.806412,-71.750416,125.556828,28.963397,48.360698,-1.504955,10.019834,1
8,39627739384253125,62.130063,-18.945104,81.075167,21.620094,38.098522,0.054528,9.603887,1
9,39627739384253866,74.893418,-83.912603,158.806021,63.193902,50.659610,1.160900,10.556721,3


In [8]:
spectra_data    = fits.open('/Users/hyp0515/data/0715_Spring_BGS_ALL_trimmed.fits')
color_data      = fits.open('/Users/hyp0515/data/0715_Spring_half_BGS_BRIGHT_catalog_with_Flux.fits')
cigale_data     = fits.open('/Users/hyp0515/data/IronPhysProp_v1.2_extracted.fits')
fastspecfit     = fits.open('/Users/hyp0515/data/0715_Spring_half_BGS_BRIGHT_catalog_fastspecfit.fits')

ids = read_ids('dp_samples_ids.txt')

SPECTRA = Spectrum(spectra_data, cigale_data, fastspecfit, load_targetID=ids)

In [9]:
dp_sample['RA'] = dp_sample['TARGETID'].map(SPECTRA.df.set_index('TARGETID')['RA'])
dp_sample['DEC'] = dp_sample['TARGETID'].map(SPECTRA.df.set_index('TARGETID')['DEC'])

In [10]:
dp_sample.head()

,TARGETID,dv_r,dv_l,delta_dv,sigma_r,sigma_l,LOGSFR,LOGM,Regime,RA,DEC
0,39627739380056564,54.235580,-77.214870,131.450450,50.133639,31.151569,0.941032,10.452706,1,177.019312,-1.965308
1,39627739380057506,120.705683,-57.385546,178.091228,39.709895,73.005164,-0.880754,10.324258,4,177.058476,-1.984356
2,39627739380058467,87.790162,-28.505013,116.295175,30.665694,51.626586,1.031611,10.402426,1,177.097982,-1.888290
3,39627739380060210,76.571222,-90.530039,167.101261,52.715443,54.794006,0.039087,10.496312,3,177.177947,-1.988386
4,39627739380061198,88.170157,-73.633545,161.803702,38.336279,50.796152,0.903058,9.514435,3,177.216425,-1.948593


In [11]:
dp_sample.to_csv('./dp_catalog.csv', index=False)